In [1]:
# SETTINGS — first run: audit only (no GPU).
PROJECT_ROOT_OVERRIDE = ""     # Optional: absolute path to sci5-wbc
TESTB_ROOT_OVERRIDE = ""       # Optional: path to extracted TestB folder
TESTB_LABEL_CSV = ""           # Optional, AUTHORITATIVE labels with relative_path,label
RUN_TESTB_INFERENCE = True     # Audit already passed (2119 labeled images); inference allowed after re-check
SEEDS = (42, 1, 7)
RUN_MODELS = ("convnext_tiny", "resnet50", "retfound_fullft")
BATCH_SIZE = {"convnext_tiny":2,"resnet50":2,"retfound_fullft":1}
TRUST_OWN_CHECKPOINTS = True   # Only for trusted, self-created local .pt files
ALLOW_PARTIAL_LABELED_EVALUATION = False  # Default refuses incomplete/incomparable labels
print('Inference enabled:',RUN_TESTB_INFERENCE)

Inference enabled: True


In [2]:
import os, json, csv, hashlib, gc, re, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

start=Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE.strip() else Path.cwd()
R=next((p for p in (start,*start.parents) if (p/'data'/'external').is_dir() and (p/'outputs'/'final').is_dir()),None)
if R is None: raise FileNotFoundError('Cannot find sci5-wbc. Set PROJECT_ROOT_OVERRIDE.')
R=R.resolve()
RAA=R/'data'/'external'/'Raabin_WBC'
OUT=R/'review_results_raabin_testb_v1';OUT.mkdir(parents=True,exist_ok=True)
EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
CANONICAL=('basophil','eosinophil','lymphocyte','monocyte','neutrophil')
LABELS={'lymphocyte':'LY','lymphocytes':'LY','lymph':'LY',
        'neutrophil':'N','neutrophils':'N','neut':'N',
        'basophil':'BA','basophils':'BA',
        'eosinophil':'EO','eosinophils':'EO',
        'monocyte':'MO','monocytes':'MO'}
PUBLISHED={'Train': {'BA':212,'EO':744,'LY':2427,'MO':561,'N':6231},
           'TestA': {'BA':89,'EO':322,'LY':1034,'MO':234,'N':2660},
           'TestB': {'BA':0,'EO':0,'LY':148,'MO':0,'N':1971}}
print('PROJECT:',R,'\nRAABIN:',RAA,'\nOUTPUT:',OUT)
if not RAA.is_dir():raise FileNotFoundError('Raabin_WBC directory not found: '+str(RAA))

def files_under(root):
    if root is None or not root.is_dir():return []
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in EXT and '__MACOSX' not in p.parts)

def class_from_parent(path, root):
    # Labels MUST come from matching class directory names; never infer from image filename.
    hits=[LABELS[k.lower()] for k in path.relative_to(root).parts[:-1] if k.lower() in LABELS]
    return hits[-1] if len(set(hits))==1 else None

def get_testb_root():
    if TESTB_ROOT_OVERRIDE.strip():
        p=Path(TESTB_ROOT_OVERRIDE).expanduser()
        if not p.is_dir():raise FileNotFoundError('TESTB_ROOT_OVERRIDE path is missing: '+str(p))
        return p.resolve()
    choices=[p for p in RAA.iterdir() if p.is_dir() and re.sub(r'[^a-z0-9]','',p.name.lower())=='testb']
    if len(choices)==1:return choices[0].resolve()
    if len(choices)>1:raise RuntimeError('Multiple TestB folders; set TESTB_ROOT_OVERRIDE: '+str(choices))
    print('TestB folder not detected under Raabin_WBC. Existing folders:',[p.name for p in RAA.iterdir() if p.is_dir()])
    return None

TESTB=get_testb_root()
print('TESTB:',TESTB)

PROJECT: C:\Users\sarah\Desktop\scı5-wbc 
RAABIN: C:\Users\sarah\Desktop\scı5-wbc\data\external\Raabin_WBC 
OUTPUT: C:\Users\sarah\Desktop\scı5-wbc\review_results_raabin_testb_v1
TESTB: C:\Users\sarah\Desktop\scı5-wbc\data\external\Raabin_WBC\TestB


In [3]:
# 1. Published-vs-local Train/TestA counts (same class-folder and file verification logic).

def verify_image(p):
    try:
        with Image.open(p) as im: im.verify()
        return None
    except Exception as e:return f'{type(e).__name__}: {e}'

split_rows=[];bad_rows=[]
for split in ('Train','TestA'):
    base=RAA/split
    images=files_under(base)
    counts=Counter()
    for p in images:
        cls=class_from_parent(p,base)
        issue=verify_image(p)
        if not cls or issue:
            bad_rows.append({'subset':split,'relative_path':p.relative_to(R).as_posix(),
                             'problem':issue or 'NO_EXACT_CLASS_PARENT'})
        else: counts[cls]+=1
    for cls in ('BA','EO','LY','MO','N'):
        split_rows.append({'subset':split,'class':cls,'local_verified':counts[cls],
                           'published':PUBLISHED[split][cls],
                           'matches_published':counts[cls]==PUBLISHED[split][cls]})
    print(split,'valid:',sum(counts.values()),'published:',sum(PUBLISHED[split].values()),
          'counts:',dict(counts),'invalid/unlabeled:',sum(x['subset']==split for x in bad_rows))
split_df=pd.DataFrame(split_rows)
split_df.to_csv(OUT/'raabin_train_testa_split_counts_v1.csv',index=False,encoding='utf-8-sig')
display(split_df)

Train valid: 10175 published: 10175 counts: {'BA': 212, 'EO': 744, 'LY': 2427, 'MO': 561, 'N': 6231} invalid/unlabeled: 0
TestA valid: 4339 published: 4339 counts: {'BA': 89, 'EO': 322, 'LY': 1034, 'MO': 234, 'N': 2660} invalid/unlabeled: 0


,subset,class,local_verified,published,matches_published
0,Train,BA,212,212,True
1,Train,EO,744,744,True
2,Train,LY,2427,2427,True
3,Train,MO,561,561,True
4,Train,N,6231,6231,True
5,TestA,BA,89,89,True
6,TestA,EO,322,322,True
7,TestA,LY,1034,1034,True
8,TestA,MO,234,234,True
9,TestA,N,2660,2660,True


In [4]:
# 2. TestB inventory. Provider-approved labels may alternatively be in a CSV.
# CSV columns: relative_path,label ; relative_path relative to TESTB directory.
# Do NOT manually assign labels from model predictions.

csv_label_map={}
if TESTB_LABEL_CSV.strip():
    table=pd.read_csv(TESTB_LABEL_CSV,dtype=str,keep_default_na=False)
    if not {'relative_path','label'}.issubset(table.columns):
        raise ValueError('TESTB_LABEL_CSV needs: relative_path,label')
    for _,rec in table.iterrows():
        key=str(rec['relative_path']).replace('\\','/').lstrip('./')
        label=LABELS.get(str(rec['label']).strip().lower(),str(rec['label']).strip().upper())
        if label not in {'BA','EO','LY','MO','N'}:raise ValueError(f'Unknown label in CSV: {rec["label"]}')
        if key in csv_label_map: raise ValueError(f'Duplicate label path in CSV: {key}')
        csv_label_map[key]=label

records=[]
for p in files_under(TESTB):
    rel=p.relative_to(TESTB).as_posix()
    folder_label=class_from_parent(p,TESTB)
    csv_label=csv_label_map.get(rel)
    if folder_label and csv_label and folder_label!=csv_label:
        status='LABEL_CONFLICT'
    elif not folder_label and not csv_label:
        status='UNLABELED'
    elif csv_label and folder_label == csv_label:
        status='LABELED_FROM_BOTH'
    elif csv_label:
        status='LABELED_FROM_VERIFIED_CSV'
    else:
        status='LABELED_FROM_CLASS_DIRECTORY'
    error=verify_image(p)
    if error:status='UNREADABLE'
    records.append({'relative_path':rel,'class':csv_label or folder_label or '',
                    'status':status,'read_error':error or ''})
manifest=pd.DataFrame(records,columns=['relative_path','class','status','read_error'])
manifest.to_csv(OUT/'testb_image_manifest_v1.csv',index=False,encoding='utf-8-sig')
valid_labels={'LABELED_FROM_BOTH','LABELED_FROM_VERIFIED_CSV','LABELED_FROM_CLASS_DIRECTORY'}
valid=manifest[manifest.status.isin(valid_labels)]
counts=valid['class'].value_counts().to_dict()
status_counts=manifest.status.value_counts().to_dict()
exact_reference=(len(valid)==2119 and counts.get('LY',0)==148 and counts.get('N',0)==1971
                 and set(counts).issubset({'LY','N'}) and len(valid)==len(manifest))
AUDIT_STATUS=('AUDIT_PASS_REFERENCE_DISTRIBUTION' if exact_reference else
              'AUDIT_NOT_READY__TESTB_MISSING' if TESTB is None else
              'AUDIT_REVIEW_INCOMPLETE_OR_DIFFERENT_LABEL_SET')
preflight={'root':str(R),'testb_directory':str(TESTB) if TESTB else None,
           'n_image_candidates':len(manifest),'n_labeled_readable':len(valid),
           'status_counts':status_counts,'class_counts':counts,
           'reference_distribution':PUBLISHED['TestB'],
           'testb_status':AUDIT_STATUS,
           'split_counts_match_source':bool(split_df.matches_published.all()),
           'five_class_testb_macro_f1_permitted':False,
           'inference_requested':bool(RUN_TESTB_INFERENCE)}
(OUT/'testb_preflight_v1.json').write_text(json.dumps(preflight,indent=2,ensure_ascii=False),encoding='utf-8')
pd.DataFrame(bad_rows).to_csv(OUT/'raabin_train_testa_unlabeled_unreadable_v1.csv',index=False,encoding='utf-8-sig')
print(json.dumps(preflight,indent=2,ensure_ascii=False))
print('TestB sample paths:')
display(manifest.head(10))

{
  "root": "C:\\Users\\sarah\\Desktop\\scı5-wbc",
  "testb_directory": "C:\\Users\\sarah\\Desktop\\scı5-wbc\\data\\external\\Raabin_WBC\\TestB",
  "n_image_candidates": 2119,
  "n_labeled_readable": 2119,
  "status_counts": {
    "LABELED_FROM_CLASS_DIRECTORY": 2119
  },
  "class_counts": {
    "N": 1971,
    "LY": 148
  },
  "reference_distribution": {
    "BA": 0,
    "EO": 0,
    "LY": 148,
    "MO": 0,
    "N": 1971
  },
  "testb_status": "AUDIT_PASS_REFERENCE_DISTRIBUTION",
  "split_counts_match_source": true,
  "five_class_testb_macro_f1_permitted": false,
  "inference_requested": true
}
TestB sample paths:


,relative_path,class,status,read_error
0,Test-B/Lymphocyte/1004.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
1,Test-B/Lymphocyte/1013.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
2,Test-B/Lymphocyte/1025.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
3,Test-B/Lymphocyte/107.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
4,Test-B/Lymphocyte/1147.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
5,Test-B/Lymphocyte/1149.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
6,Test-B/Lymphocyte/1172.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
7,Test-B/Lymphocyte/1173.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
8,Test-B/Lymphocyte/119.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,
9,Test-B/Lymphocyte/1191.jpg,LY,LABELED_FROM_CLASS_DIRECTORY,


## 3. Optional checkpoint inference — only after label audit

Default `RUN_TESTB_INFERENCE = False` keeps this notebook CPU-only. If the local TestB dataset does not contain 2,119 correctly labeled/readable images, obtain the original double-labeled cropped TestB partition from the dataset provider. If you have a provider-approved label CSV, set `TESTB_LABEL_CSV` first. Do not infer labels from model predictions.

When the audit passes, change `RUN_TESTB_INFERENCE = True` in the first settings cell and restart/run all. The inference uses the same 224×224 RGB/normalization and fixed training checkpoints. On a 4-GB GPU, RETFound may take substantially longer than the audit; fan activity can increase. The code refuses five-class TestB macro-F1 reporting.

In [5]:
# Optional stage dependencies are imported only if inference is enabled.
if RUN_TESTB_INFERENCE:
    if not (exact_reference or ALLOW_PARTIAL_LABELED_EVALUATION):
        raise RuntimeError('TestB reference-label audit did not PASS. No classification metrics computed.')
    if len(valid)==0 or not {'LY','N'}.issubset(set(valid['class'])):
        raise RuntimeError('Need both expert-labeled LY and N. No metrics computed.')
    import torch, torch.nn as nn
    from torch.utils.data import Dataset,DataLoader
    from torchvision import models,transforms
    from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
    import sys
    if not (R/'RETFound'/'models_vit.py').is_file():raise FileNotFoundError('RETFound/models_vit.py missing')
    sys.path.insert(0,str(R/'RETFound'))
    import models_vit
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    FULL=['BA','BL','BNE','EO','LY','MMY','MO','MY','PC','PLY','PMY','SNE','VLY']
    FIVE=['BA','EO','LY','MO','SNE']
    FIVE_IDS=[FULL.index(c) for c in FIVE]
    IDX_LY=FIVE.index('LY');IDX_N=FIVE.index('SNE')
    tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
                            transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    class TestBData(Dataset):
        def __init__(self,items):
            self.items=[(TESTB/row.relative_path,row.class_label,row.relative_path) for row in items.itertuples(index=False)]
        def __len__(self):return len(self.items)
        def __getitem__(self,idx):
            path,cl,rel=self.items[idx]
            with Image.open(path) as im: x=tf(im.convert('RGB'))
            return x,(0 if cl=='LY' else 1),rel
    # Authoritatively labeled test-B images only, in stable sorted path order.
    valid_sorted=valid.sort_values('relative_path').rename(columns={'class':'class_label'}).reset_index(drop=True)
    DS=TestBData(valid_sorted)
    print('Device:',DEVICE,'labelled testB images:',len(DS))

C:\Users\sarah\.conda\envs\medxai_gpu\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Device: cuda labelled testB images: 2119


In [6]:
if RUN_TESTB_INFERENCE:
    def checkpoint(model_name,seed):
        folder=R/'outputs'/'final'/model_name/f'seed{seed}'
        candidate=folder/f'{model_name}_seed{seed}_best.pt'
        if not candidate.is_file():
            raise FileNotFoundError(f'Checkpoint missing: {candidate}. Do not substitute other epochs.')
        return candidate

    def state_from_trusted_checkpoint(file):
        try:
            obj=torch.load(file,map_location='cpu',weights_only=True)
        except Exception:
            if not TRUST_OWN_CHECKPOINTS: raise
            obj=torch.load(file,map_location='cpu',weights_only=False)
        if isinstance(obj,dict):
            for k in ('model','state_dict','model_state_dict'):
                if isinstance(obj.get(k),dict):obj=obj[k];break
        if not isinstance(obj,dict):raise ValueError('Checkpoint state_dict not found')
        return {k.removeprefix('module.'):v for k,v in obj.items() if isinstance(k,str) and torch.is_tensor(v)}

    def make_model(name,cp):
        if name=='convnext_tiny':
            m=models.convnext_tiny(weights=None)
            m.classifier[2]=nn.Linear(m.classifier[2].in_features,13)
        elif name=='resnet50':
            m=models.resnet50(weights=None)
            m.fc=nn.Linear(m.fc.in_features,13)
        elif name=='retfound_fullft':
            m=models_vit.RETFound_mae(img_size=224,num_classes=13,drop_path_rate=0,global_pool=True)
        else:raise ValueError(name)
        m.load_state_dict(state_from_trusted_checkpoint(cp),strict=True)
        return m.to(DEVICE).eval()

    def retfound_logits(m,x):
        b=x.shape[0]
        tokens=m.patch_embed(x)
        if tokens.ndim!=3 or tokens.shape[1]!=196:raise ValueError('Invalid 224x224 patch tokens')
        tokens=torch.cat([m.cls_token.expand(b,-1,-1),tokens],dim=1)
        if tokens.shape[1]!=m.pos_embed.shape[1]:raise ValueError('Positional embeddings mismatch')
        tokens=m.pos_drop(tokens+m.pos_embed)
        for block in m.blocks:tokens=block(tokens)
        pooled=m.fc_norm(tokens[:,1:,:].mean(1))
        return m.head(pooled)

    def score_from_output(truth,pred):
        # Five-way prediction evaluated on a two-true-class subset: report ONLY per-class recall,
        # and explicitly count decisions outside LY,N. NOT a 5-class macro F1.
        truth=np.asarray(truth,dtype=int);pred=np.asarray(pred,dtype=int)
        a=[]
        for j,name in [(0,'LY'),(1,'N')]:
            mask=truth==j
            desired=IDX_LY if j==0 else IDX_N
            a.append({'class':name,'support':int(mask.sum()),
                      'recall_of_five_way_rule':float((pred[mask]==desired).mean())})
        off=~np.isin(pred,[IDX_LY,IDX_N])
        return a,int(off.sum())

    print('Strict checkpoint/model setup prepared. Inference next cell.')

Strict checkpoint/model setup prepared. Inference next cell.


In [7]:
# Optional: nine fixed-checkpoint test-B runs; never changes Train/TestA outputs.
if RUN_TESTB_INFERENCE:
    result_rows=[];class_rows=[];confusion_rows=[];issue_rows=[]
    # 'class' was renamed to 'class_label' in the preceding preparation cell.
    # Assert this before loading any model to prevent wasting GPU time on a metadata error.
    if 'class_label' not in valid_sorted.columns:
        raise RuntimeError(f"Expected class_label column, found: {valid_sorted.columns.tolist()}")
    if not valid_sorted['class_label'].isin(['LY', 'N']).all():
        raise RuntimeError('Unexpected TestB label: only LY and N are supported')
    truth_expected=np.array([0 if c=='LY' else 1 for c in valid_sorted['class_label']],dtype=int)
    if int((truth_expected==0).sum()) != 148 or int((truth_expected==1).sum()) != 1971:
        raise RuntimeError('TestB support changed since audit; stop before GPU inference')
    if len(DS) != len(truth_expected):
        raise RuntimeError('Dataset length and audited labels do not match')
    print('TestB label mapping PASS:', len(truth_expected), 'images; LY=148, N=1971')
    for model_name in RUN_MODELS:
        for seed in SEEDS:
            t0=time.time()
            try:
                cp=checkpoint(model_name,seed)
                m=make_model(model_name,cp)
                loader=DataLoader(DS,batch_size=BATCH_SIZE[model_name],shuffle=False,num_workers=0,
                                  pin_memory=(DEVICE.type=='cuda'))
                collected=[];collected_paths=[];yy=[]
                with torch.inference_mode():
                    for batch,(x,y,paths) in enumerate(loader,1):
                        out=retfound_logits(m,x.to(DEVICE)) if model_name=='retfound_fullft' else m(x.to(DEVICE))
                        if out.ndim!=2 or out.shape[1]!=13:raise RuntimeError('Expected [B,13] logits')
                        q=torch.softmax(out.float(),dim=1).cpu().numpy()
                        collected.append(q);yy.extend(y.tolist());collected_paths.extend(paths)
                        if batch%300==0:print(model_name,seed,batch,'/',len(loader))
                Q=np.concatenate(collected)
                y=np.asarray(yy,dtype=int)
                if not np.array_equal(y,truth_expected):raise RuntimeError('Inferred label order is inconsistent with audit')
                if list(collected_paths)!=list(valid_sorted.relative_path):raise RuntimeError('Path/order mismatch')
                # Five-class original rule and BNE+SNE alternative.
                p5=Q[:,FIVE_IDS].argmax(1).astype(int)
                m5=np.column_stack([Q[:,FULL.index(k)] for k in ['BA','EO','LY','MO']]+[
                                    Q[:,FULL.index('BNE')]+Q[:,FULL.index('SNE')]])
                p5plus=m5.argmax(1).astype(int)
                # TWO-class conditional rule (LY and N only): independent narrower task.
                binQ=Q[:,[FULL.index('LY'),FULL.index('SNE')]]
                pbin=binQ.argmax(1).astype(int)
                binPlus=np.column_stack([Q[:,FULL.index('LY')],Q[:,FULL.index('BNE')]+Q[:,FULL.index('SNE')]])
                pbinPlus=binPlus.argmax(1).astype(int)
                for mapping, pp5, ppbin in [('five_output_baseline',p5,pbin),('BNE_plus_SNE',p5plus,pbinPlus)]:
                    recalls,off=score_from_output(y,pp5)
                    for r in recalls:class_rows.append(dict(model=model_name,seed=seed,mapping=mapping,**r))
                    n_ly=int((y==0).sum());n_n=int((y==1).sum())
                    result_rows.append(dict(model=model_name,seed=seed,mapping=mapping,n=len(y),
                        support_LY=n_ly,support_N=n_n,
                        five_output_recall_LY=recalls[0]['recall_of_five_way_rule'],
                        five_output_recall_N=recalls[1]['recall_of_five_way_rule'],
                        five_output_off_class_decisions=off,
                        conditional_two_class_macro_f1=float(f1_score(y,ppbin,average='macro',labels=[0,1],zero_division=0)),
                        conditional_two_class_balanced_accuracy=float(balanced_accuracy_score(y,ppbin)),
                        conditional_two_class_accuracy=float(np.mean(y==ppbin)),
                        checkpoint=str(cp),seconds=round(time.time()-t0,2)))
                    for i,true_cls in enumerate(('LY','N')):
                        for j,pred_cls in enumerate(('BA','EO','LY','MO','N')):
                            confusion_rows.append(dict(model=model_name,seed=seed,mapping=mapping,
                                                      true_class=true_cls,predicted_class=pred_cls,
                                                      count=int(np.count_nonzero((y==i)&(pp5==j)))))
                savedir=OUT/'predictions'/model_name/f'seed{seed}';savedir.mkdir(parents=True,exist_ok=True)
                np.savez_compressed(savedir/'testb_probabilities_and_labels.npz',q13=Q,truth_LY0_N1=y,
                                    pred5=p5,pred5_BNEplusSNE=p5plus,
                                    pred_binary=pbin,pred_binary_BNEplusSNE=pbinPlus)
                pd.DataFrame({'relative_path':collected_paths,'truth':[('LY','N')[k] for k in y],
                              'pred5':[('BA','EO','LY','MO','N')[k] for k in p5],
                              'pred5_BNEplusSNE':[('BA','EO','LY','MO','N')[k] for k in p5plus],
                              'pred2':[('LY','N')[k] for k in pbin],
                              'pred2_BNEplusSNE':[('LY','N')[k] for k in pbinPlus]})\
                  .to_csv(savedir/'testb_sample_predictions.csv',index=False,encoding='utf-8-sig')
                print('COMPLETED',model_name,'seed',seed,'n',len(y),'seconds',round(time.time()-t0,1))
            except Exception as exc:
                issue_rows.append({'model':model_name,'seed':seed,'error':type(exc).__name__+': '+str(exc)})
                print('ERROR:',model_name,seed,repr(exc))
            finally:
                if 'm' in locals():del m
                gc.collect()
                if DEVICE.type=='cuda':torch.cuda.empty_cache()
    results=pd.DataFrame(result_rows)
    results.to_csv(OUT/'testb_two_class_sensitivity_per_seed_v1.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(class_rows).to_csv(OUT/'testb_lymph_neut_recall_v1.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(confusion_rows).to_csv(OUT/'testb_2x5_confusion_v1.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(issue_rows,columns=['model','seed','error']).to_csv(OUT/'testb_inference_issues_v1.csv',index=False,encoding='utf-8-sig')
    if len(results):
        summary=results.groupby(['model','mapping'],as_index=False).agg(
            n_seeds=('seed','nunique'),support_LY=('support_LY','first'),support_N=('support_N','first'),
            mean_recall_LY=('five_output_recall_LY','mean'),mean_recall_N=('five_output_recall_N','mean'),
            mean_conditional_two_class_macro_f1=('conditional_two_class_macro_f1','mean'),
            sample_sd_conditional_two_class_macro_f1=('conditional_two_class_macro_f1','std'))
        summary.to_csv(OUT/'testb_two_class_sensitivity_summary_v1.csv',index=False,encoding='utf-8-sig')
        display(summary)
else:
    print('AUDIT-ONLY RUN. No GPU inference, no fabricated TestB performance. To evaluate, require PASS and explicitly set RUN_TESTB_INFERENCE=True.')

TestB label mapping PASS: 2119 images; LY=148, N=1971
convnext_tiny 42 300 / 1060
convnext_tiny 42 600 / 1060
convnext_tiny 42 900 / 1060
COMPLETED convnext_tiny seed 42 n 2119 seconds 28.1
convnext_tiny 1 300 / 1060
convnext_tiny 1 600 / 1060
convnext_tiny 1 900 / 1060
COMPLETED convnext_tiny seed 1 n 2119 seconds 20.1
convnext_tiny 7 300 / 1060
convnext_tiny 7 600 / 1060
convnext_tiny 7 900 / 1060
COMPLETED convnext_tiny seed 7 n 2119 seconds 19.9
resnet50 42 300 / 1060
resnet50 42 600 / 1060
resnet50 42 900 / 1060
COMPLETED resnet50 seed 42 n 2119 seconds 41.7
resnet50 1 300 / 1060
resnet50 1 600 / 1060
resnet50 1 900 / 1060
COMPLETED resnet50 seed 1 n 2119 seconds 40.9
resnet50 7 300 / 1060
resnet50 7 600 / 1060
resnet50 7 900 / 1060
COMPLETED resnet50 seed 7 n 2119 seconds 46.6
retfound_fullft 42 300 / 2119
retfound_fullft 42 600 / 2119
retfound_fullft 42 900 / 2119
retfound_fullft 42 1200 / 2119
retfound_fullft 42 1500 / 2119
retfound_fullft 42 1800 / 2119
retfound_fullft 42 2100

,model,mapping,n_seeds,support_LY,support_N,mean_recall_LY,mean_recall_N,mean_conditional_two_class_macro_f1,sample_sd_conditional_two_class_macro_f1
0,convnext_tiny,BNE_plus_SNE,3,148,1971,0.907658,0.883815,0.882883,0.057204
1,convnext_tiny,five_output_baseline,3,148,1971,0.907658,0.881448,0.879355,0.057093
2,resnet50,BNE_plus_SNE,3,148,1971,0.914414,0.880602,0.897739,0.063830
3,resnet50,five_output_baseline,3,148,1971,0.914414,0.876205,0.891266,0.062038
4,retfound_fullft,BNE_plus_SNE,3,148,1971,0.966216,0.843565,0.724773,0.091950
5,retfound_fullft,five_output_baseline,3,148,1971,0.966216,0.838661,0.719324,0.090175
